# Track1 End-to-End Learner Notebook

이 노트북은 **Track1 실습(Mission 1~5)** 전체를 한 번에 따라갈 수 있도록 구성된 참가자용 워크벤치입니다.

- Mission 1: 비즈니스 질문 정리
- Mission 2: 데이터 구조 읽기와 품질 개념 이해
- Mission 3: 표준 스키마/코드 정규화
- Mission 4: Ontology 엔터티/관계 설계
- Mission 5: 매핑/의미 경로 확인 + Track2 인계 패키지 초안


## Fabric Notebook 업로드/실행 안내

1. Fabric Workspace에서 **New → Notebook → Import notebook**으로 이 파일을 업로드합니다.
2. Notebook을 연 뒤 세션을 시작하고, 셀을 위에서 아래 순서대로 실행합니다.
3. Mission 4/Appendix 실습 시 Ontology 단계는 구조/관계/매핑 구성 작업이며, 테스트 데이터 자체는 별도 적재 단계에서 준비됩니다.

## Track1 기술 맥락과 참조 문서

Track1의 기술 목표는 정형 원천 데이터를 **의미 모델(Ontology)** 로 승격해 FabricIQ 질의의 안정성을 확보하는 것입니다.

- 통합 설계 기준: [Microsoft_IQ_Workshop_Integrated_Plan.md](../common/docs/Microsoft_IQ_Workshop_Integrated_Plan.md)
- 트랙 실습 기준(DoD/미션): [WORKBOOK.md](./WORKBOOK.md)
- 데이터 구조/관계 상세: [Track1_Data_Structure_Detailed_Guide.md](./docs/Track1_Data_Structure_Detailed_Guide.md)
- 온톨로지 모델링 개념: [Track1_Ontology_Concepts_and_Graph_Design_Guide.md](./docs/Track1_Ontology_Concepts_and_Graph_Design_Guide.md)
- 데이터셋 계약/품질 규칙: [track1/data/README.md](./data/README.md)

핵심 기술 축:
1. **질문 중심 모델링**: Q1~Q5를 엔터티/관계 경로로 고정
2. **품질 경계 이해**: P1 품질 개념은 설명으로 익히고 표준화·매핑에 집중
3. **그래프화**: 물리 FK + 논리(다중 홉) 관계를 함께 설계
4. **검증 가능성**: SQL 기준값으로 온톨로지 경로의 재현성 확보


In [ ]:
from __future__ import annotations

from pathlib import Path
from dataclasses import dataclass
import csv
import json
import sqlite3

def find_track1_data_root() -> Path:
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        if (base / "customers.csv").exists() and (base / "orders.csv").exists():
            return base
        candidate = base / "track1" / "data"
        if (candidate / "customers.csv").exists() and (candidate / "orders.csv").exists():
            return candidate
    raise FileNotFoundError("track1/data 루트를 찾을 수 없습니다.")

DATA_ROOT = find_track1_data_root()
WORKBENCH_DIR = DATA_ROOT / "generated" / "workbench"
WORKBENCH_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATA_ROOT: {DATA_ROOT}")
print(f"WORKBENCH_DIR: {WORKBENCH_DIR}")


## Mission 1. 비즈니스 질문 정리 (Q1~Q5)

Track1 공통 질문을 확정하고 질문별 핵심 테이블을 연결합니다.

기술 포인트:
- 질문을 KPI와 데이터 경로로 분해해야 이후 Ontology 관계가 흔들리지 않습니다.
- 질문-테이블 매핑은 Track4 FoundryIQ의 도구 라우팅 기준이 됩니다.

관련 문서:
- [WORKBOOK.md](./WORKBOOK.md)
- [Track1_Data_Structure_Detailed_Guide.md#advanced-scenario](./docs/Track1_Data_Structure_Detailed_Guide.md#advanced-scenario)
- [Track1_Data_Structure_Detailed_Guide.md#source-table-structure](./docs/Track1_Data_Structure_Detailed_Guide.md#source-table-structure)


In [ ]:
question_map = [
    {
        "id": "Q1",
        "question": "결제 실패가 캠페인 전환율에 미치는 영향은 무엇인가?",
        "tables": ["campaigns", "campaign_attribution", "customers", "orders", "payments"],
    },
    {
        "id": "Q2",
        "question": "배송 지연은 반품률과 고객 만족도에 어떤 영향을 미치는가?",
        "tables": ["shipments", "returns", "support_tickets", "orders", "customers", "channels"],
    },
    {
        "id": "Q3",
        "question": "프로모션 유형별 할인 전략이 매출총이익과 재구매율에 미치는 영향은 무엇인가?",
        "tables": ["promotions", "order_promotions", "orders", "order_items", "products", "customers"],
    },
    {
        "id": "Q4",
        "question": "재고 부족/품절 경험은 주문 취소율과 고객센터 문의량에 어떤 영향을 미치는가?",
        "tables": ["inventory_snapshots", "products", "orders", "support_tickets", "channels"],
    },
    {
        "id": "Q5",
        "question": "채널·고객등급별 반품 사유 패턴은 재구매율에 어떤 차이를 만드는가?",
        "tables": ["returns", "orders", "customers", "channels", "order_items", "products"],
    },
]

for row in question_map:
    print(f"[{row['id']}] {row['question']}")
    print("  tables:", ", ".join(row["tables"]))

(WORKBENCH_DIR / "mission1_question_map.json").write_text(
    json.dumps(question_map, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("\nSaved:", WORKBENCH_DIR / "mission1_question_map.json")


## Mission 2. 데이터 구조 읽기와 품질 개념 이해

CSV 14개를 SQLite 메모리 DB로 로드하고 행 수·필수 키·질문별 사용 테이블을 확인합니다.

기술 포인트:
- 참조 무결성·중복·결측·이상값은 KPI와 Ontology 경로를 왜곡할 수 있다는 개념만 이해합니다.
- P1 오류 위치·건수를 찾거나 수정하는 쿼리는 참가자 실습과 완료 기준에서 제외합니다.
- 표준화가 필요한 코드와 질문별 핵심 키는 다음 미션의 매핑 설계에 사용합니다.

관련 문서:
- [track1/data/README.md](./data/README.md)
- [Track1_Data_Structure_Detailed_Guide.md#profiling-checkpoints](./docs/Track1_Data_Structure_Detailed_Guide.md#profiling-checkpoints)
- [WORKBOOK.md](./WORKBOOK.md)


In [ ]:
TABLE_FILES = {
    "customers": "customers.csv",
    "products": "products.csv",
    "orders": "orders.csv",
    "order_items": "order_items.csv",
    "returns": "returns.csv",
    "channels": "channels.csv",
    "payments": "payments.csv",
    "shipments": "shipments.csv",
    "inventory_snapshots": "inventory_snapshots.csv",
    "promotions": "promotions.csv",
    "order_promotions": "order_promotions.csv",
    "campaigns": "campaigns.csv",
    "campaign_attribution": "campaign_attribution.csv",
    "support_tickets": "support_tickets.csv",
}

conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row

def quote_ident(name: str) -> str:
    return '"' + name.replace('"', '""') + '"'

row_count = {}
for table, filename in TABLE_FILES.items():
    path = DATA_ROOT / filename
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        cols = reader.fieldnames or []
        col_defs = ", ".join(f"{quote_ident(c)} TEXT" for c in cols)
        conn.execute(f"CREATE TABLE {quote_ident(table)} ({col_defs});")
        placeholders = ", ".join(["?"] * len(cols))
        insert_sql = f"INSERT INTO {quote_ident(table)} ({', '.join(quote_ident(c) for c in cols)}) VALUES ({placeholders});"
        rows = [tuple((r.get(c) if r.get(c) != "" else None) for c in cols) for r in reader]
        conn.executemany(insert_sql, rows)
        row_count[table] = len(rows)

print("Loaded tables:")
for t in sorted(row_count):
    print(f"- {t}: {row_count[t]}")


In [ ]:
def run_query(sql: str) -> list[dict[str, object]]:
    cur = conn.execute(sql)
    return [dict(r) for r in cur.fetchall()]

data_orientation = {
    "tableCount": len(row_count),
    "rowCounts": row_count,
    "questionKeyMap": {
        "Q1": ["campaign_id", "order_id", "payment_status"],
        "Q2": ["order_id", "shipment_status", "return_id", "ticket_type"],
        "Q3": ["promotion_id", "order_id", "customer_id"],
        "Q4": ["product_id", "order_id", "on_hand_qty"],
        "Q5": ["customer_id", "channel_id", "return_reason"],
    },
    "qualityConcepts": [
        {"concept": "referential-integrity", "impact": "ontology path can break", "exercise": "explanation-only"},
        {"concept": "duplicate-key", "impact": "aggregates can be overstated", "exercise": "explanation-only"},
        {"concept": "missing-value", "impact": "classification can be incomplete", "exercise": "explanation-only"},
        {"concept": "outlier", "impact": "metrics can be distorted", "exercise": "explanation-only"},
    ],
    "p1ValidationExecuted": False,
}

print(json.dumps(data_orientation, ensure_ascii=False, indent=2))

(WORKBENCH_DIR / "mission2_data_orientation.json").write_text(
    json.dumps(data_orientation, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("\nSaved:", WORKBENCH_DIR / "mission2_data_orientation.json")


## Mission 3. 표준 스키마/코드 정규화 설계

비표준 상태값을 표준 코드셋으로 변환하는 매핑 초안을 만듭니다.

기술 포인트:
- 표준화는 단순 치환이 아니라, 분석 단위(상태/재시도/시점)를 분리해 의미를 보존하는 과정입니다.
- 키 규칙(`*_id`), 타입 규칙(Date/DateTime), 코드 규칙을 동시에 맞춰야 Ontology 속성 충돌이 줄어듭니다.

관련 문서:
- [Track1_Data_Structure_Detailed_Guide.md#standardization-rules](./docs/Track1_Data_Structure_Detailed_Guide.md#standardization-rules)
- [WORKBOOK.md](./WORKBOOK.md)


### Mission 3 체크포인트

- `orders.order_status`, `payments.payment_status`, `shipments.shipment_status`의 정상 코드값 분포를 확인합니다. P1 결측 탐지는 제외합니다.
- 표준 코드셋으로 매핑할 때 **의미가 섞인 값**(예: `RetrySuccess`)은 상태/재시도 여부를 분리해 기록합니다.
- Track2에서 같은 키워드/상태를 사용하므로, 표준화 규칙은 인계 패키지에 반드시 포함합니다.

참조:
- [Track1_Data_Structure_Detailed_Guide.md#mapping-3step](./docs/Track1_Data_Structure_Detailed_Guide.md#mapping-3step)
- [PREREQUISITES.md](../track2/PREREQUISITES.md)


In [ ]:
ORDER_STATUS_MAP = {
    "Completed": "PAID",
    "Cancelled": "CANCELLED",
    "NEW": "NEW",
    "PAID": "PAID",
    "SHIPPED": "SHIPPED",
    "RETURNED": "RETURNED",
}

PAYMENT_STATUS_MAP = {
    "Success": "AUTHORIZED",
    "RetrySuccess": "AUTHORIZED",
    "Failed": "FAILED",
    "INITIATED": "INITIATED",
    "AUTHORIZED": "AUTHORIZED",
    "FAILED": "FAILED",
    "REFUNDED": "REFUNDED",
}

SHIPMENT_STATUS_MAP = {
    "Delivered": "DELIVERED",
    "Delayed": "DELAYED",
    "InTransit": "IN_TRANSIT",
    "READY": "READY",
    "DELIVERED": "DELIVERED",
    "DELAYED": "DELAYED",
}

def status_distribution(table: str, col: str) -> list[tuple[str, int]]:
    rows = run_query(
        f"SELECT {col} AS v, COUNT(*) AS c FROM {table} WHERE {col} IS NOT NULL AND TRIM({col}) <> '' GROUP BY {col} ORDER BY c DESC"
    )
    return [(str(r['v']), int(r['c'])) for r in rows]

for table, col, mapping in [
    ("orders", "order_status", ORDER_STATUS_MAP),
    ("payments", "payment_status", PAYMENT_STATUS_MAP),
    ("shipments", "shipment_status", SHIPMENT_STATUS_MAP),
]:
    print(f"\n[{table}.{col}] raw distribution")
    dist = status_distribution(table, col)
    for raw, cnt in dist:
        print(f"- {raw}: {cnt} -> {mapping.get(raw, '<UNMAPPED>')}")

standardization_plan = {
    "key_rule": "<entity>_id (snake_case)",
    "datetime_rule": "DATE/TIMESTAMP type normalization",
    "order_status_map": ORDER_STATUS_MAP,
    "payment_status_map": PAYMENT_STATUS_MAP,
    "shipment_status_map": SHIPMENT_STATUS_MAP,
}
(WORKBENCH_DIR / "mission3_standardization_plan.json").write_text(
    json.dumps(standardization_plan, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("\nSaved:", WORKBENCH_DIR / "mission3_standardization_plan.json")


## Mission 4. Ontology 엔터티/관계 설계

권장 기준(엔터티 14개, 관계 20개 + 확장 논리관계)을 구조화합니다.

기술 포인트:
- 물리 테이블 관계(FK)와 의미 관계(논리 경로)를 분리해 모델링해야 설명 가능성이 높아집니다.
- Track3 Tool 결합 질의를 고려해 경로 기반(예: Campaign->Order->Payment->Shipment->Return)으로 설계합니다.

관련 문서:
- [Track1_Data_Structure_Detailed_Guide.md#ontology-model](./docs/Track1_Data_Structure_Detailed_Guide.md#ontology-model)
- [Track1_Ontology_Concepts_and_Graph_Design_Guide.md](./docs/Track1_Ontology_Concepts_and_Graph_Design_Guide.md)
- [Track1_Instructor_Script_v1.0.md](./docs/Track1_Instructor_Script_v1.0.md)


### Mission 4 설계 원칙

- 엔터티는 명사형, 관계는 동사형으로 작성합니다.
- 물리 FK 관계와 논리(다중 홉) 관계를 구분해 기록합니다.
- 강사 대본 연계 확장 관계도 함께 포함합니다.
  - `Payment relates_to Return (logical)`
  - `Shipment relates_to Return (logical)`
  - `Order applies Promotion (logical path)`
- 핵심 경로 예시: `Campaign -> Order -> Payment -> Shipment -> Return`


In [ ]:
entities = [
    "Customer", "Product", "Channel", "Campaign", "Promotion", "Order", "OrderItem",
    "Payment", "Shipment", "Return", "SupportTicket", "InventorySnapshot",
    "OrderPromotion", "CampaignAttribution",
]

relationships_core = [
    "Customer places Order",
    "Order belongs_to Channel",
    "Order has Payment",
    "Order fulfilled_by Shipment",
    "Order includes OrderItem",
    "OrderItem references Product",
    "Order has Return",
    "Return references Product",
    "Return requested_by Customer",
    "Customer raises SupportTicket",
    "SupportTicket relates_to Order",
    "Product has InventorySnapshot",
    "Order receives OrderPromotion",
    "OrderPromotion points_to Promotion",
    "Campaign drives CampaignAttribution",
    "CampaignAttribution points_to Order",
    "CampaignAttribution points_to Customer",
    "Promotion influences Order (logical)",
    "Campaign influences Order (logical)",
    "Customer purchases Product (logical)",
]

relationships_extension = [
    "Payment relates_to Return (logical)",
    "Shipment relates_to Return (logical)",
    "Order applies Promotion (logical path)",
]

print("entity_count:", len(entities))
print("core_relationship_count:", len(relationships_core))
print("extension_relationship_count:", len(relationships_extension))
print("\nDoD range check (10-16 entities, 15-25 relationships):")
print("- entities_ok:", 10 <= len(entities) <= 16)
print("- relationships_ok:", 15 <= len(relationships_core) <= 25)

(WORKBENCH_DIR / "mission4_entities.json").write_text(
    json.dumps(entities, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
(WORKBENCH_DIR / "mission4_relationships_core.json").write_text(
    json.dumps(relationships_core, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
(WORKBENCH_DIR / "mission4_relationships_extension.json").write_text(
    json.dumps(relationships_extension, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("\nSaved mission4 artifacts under", WORKBENCH_DIR)


In [ ]:
mapping_template_csv = WORKBENCH_DIR / "mission5_mapping_template.csv"
with mapping_template_csv.open("w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["entity", "source_table", "source_column", "standard_column", "ontology_property", "note"])
    writer.writerow(["Order", "orders", "order_status", "order_status_std", "Order.status", "status code normalization"])
    writer.writerow(["Payment", "payments", "payment_status", "payment_status_std", "Payment.status", "RetrySuccess -> AUTHORIZED"])
    writer.writerow(["Shipment", "shipments", "shipment_status", "shipment_status_std", "Shipment.status", "CamelCase -> UPPER_SNAKE"])
    writer.writerow(["Campaign", "campaign_attribution", "campaign_id", "campaign_id", "Campaign.campaign_id", "bridge mapping"])
    writer.writerow(["SupportTicket", "support_tickets", "ticket_reason", "ticket_reason_std", "SupportTicket.reason", "controlled vocabulary"])

print("Saved mapping template:", mapping_template_csv)


## Mission 5. 매핑 검토 + Ontology 의미 경로 확인 + Track2 인계 패키지

매핑표의 핵심 엔터티를 확인한 뒤, 두 질문의 Ontology 경로와 SQL baseline을 저장합니다.

기술 포인트:
- P1 참조무결성/중복/결측/이상값 탐지 쿼리는 참가자 실습에서 실행하지 않습니다.
- Level A는 엔터티-관계 경로와 SQL baseline을 확인하고, Level B는 GraphModel 가능 환경에서만 비교합니다.

관련 문서:
- [Track1_Data_Structure_Detailed_Guide.md#semantic-validation-structure](./docs/Track1_Data_Structure_Detailed_Guide.md#semantic-validation-structure)
- [WORKBOOK.md](./WORKBOOK.md)
- [PREREQUISITES.md](../track2/PREREQUISITES.md)


In [ ]:
with mapping_template_csv.open("r", encoding="utf-8", newline="") as f:
    mapping_rows = list(csv.DictReader(f))

required_entities = {"Order", "Payment", "Shipment", "Campaign", "SupportTicket"}
mapped_entities = {row["entity"] for row in mapping_rows}
mapping_review = {
    "mappingRowCount": len(mapping_rows),
    "mappedEntities": sorted(mapped_entities),
    "requiredEntities": sorted(required_entities),
    "requiredMappingsPresent": required_entities.issubset(mapped_entities),
    "p1ValidationExecuted": False,
}

print(json.dumps(mapping_review, ensure_ascii=False, indent=2))

(WORKBENCH_DIR / "mission5_mapping_review.json").write_text(
    json.dumps(mapping_review, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("\nSaved:", WORKBENCH_DIR / "mission5_mapping_review.json")


### Mission 5. 온톨로지 의미 경로 확인 (Level A 필수 / Level B 선택)

두 시나리오의 경로와 SQL baseline을 확인합니다.

- 시나리오 A(Q1): `Campaign -> CampaignAttribution -> Order -> Payment`
- 시나리오 B(Q3): `Promotion -> OrderPromotion -> Order -> Customer`

실행 순서:
1. SQL baseline 계산(이 셀 아래 코드)
2. `getDefinition`에서 엔터티/관계 존재 확인
3. (GraphModel 가능 시) `executeQuery` 실행 후 SQL baseline과 비교

참조: [WORKBOOK.md 미션 5](./WORKBOOK.md)


In [ ]:
semantic_scenarios = {
    "A": {
        "question": "캠페인 유입 주문 중 결제 실패 주문은?",
        "path": "Campaign->CampaignAttribution->Order->Payment",
        "baseline_sql": """
            SELECT DISTINCT ca.campaign_id, o.order_id
            FROM campaign_attribution ca
            JOIN orders o ON ca.order_id = o.order_id
            JOIN payments p ON p.order_id = o.order_id
            WHERE UPPER(TRIM(COALESCE(p.payment_status, ''))) = 'FAILED'
            ORDER BY ca.campaign_id, o.order_id
            LIMIT 20;
        """,
        "graph_query": """
            MATCH (c:`Campaign`)-[:`Campaign_influences_Order`]->(o:`Order`)
            MATCH (o)-[:`Order_has_Payment`]->(p:`Payment`)
            WHERE toUpper(coalesce(p.payment_status, '')) = 'FAILED'
            RETURN c.campaign_id AS campaign_id, o.order_id AS order_id
            LIMIT 20;
        """,
        "compare_rule": "row_count 동일 + (campaign_id, order_id) 샘플 10건 동일",
    },
    "B": {
        "question": "프로모션 유형별 재구매율은?",
        "path": "Promotion->OrderPromotion->Order->Customer",
        "baseline_sql": """
            WITH customer_repeat AS (
              SELECT customer_id, CASE WHEN COUNT(*) >= 2 THEN 1 ELSE 0 END AS is_repeat
              FROM orders
              GROUP BY customer_id
            )
            SELECT
              p.promotion_type,
              COUNT(DISTINCT o.order_id) AS orders_cnt,
              AVG(cr.is_repeat) AS repurchase_rate
            FROM orders o
            JOIN order_promotions op ON o.order_id = op.order_id
            JOIN promotions p ON op.promotion_id = p.promotion_id
            LEFT JOIN customer_repeat cr ON o.customer_id = cr.customer_id
            GROUP BY p.promotion_type
            ORDER BY p.promotion_type;
        """,
        "graph_query": """
            MATCH (p:`Promotion`)-[:`OrderPromotion_points_to_Promotion`]->(op:`OrderPromotion`)
            MATCH (o:`Order`)-[:`Order_receives_OrderPromotion`]->(op)
            MATCH (c:`Customer`)-[:`Customer_places_Order`]->(o)
            MATCH (c)-[:`Customer_places_Order`]->(o2:`Order`)
            WITH p.promotion_type AS promotion_type,
                 o.order_id AS order_id,
                 c.customer_id AS customer_id,
                 COUNT(DISTINCT o2.order_id) AS customer_order_count
            RETURN promotion_type, order_id, customer_id,
                   CASE WHEN customer_order_count >= 2 THEN 1 ELSE 0 END AS is_repeat
            LIMIT 20000;
        """,
        "compare_rule": "promotion_type별 orders_cnt 차이 0 + repurchase_rate 허용오차 ±0.001",
    },
}

semantic_baseline = {}
for sid, cfg in semantic_scenarios.items():
    rows = run_query(cfg["baseline_sql"])
    semantic_baseline[sid] = {
        "question": cfg["question"],
        "path": cfg["path"],
        "baselineSqlRows": len(rows),
        "sampleRows": rows[:10],
        "compareRule": cfg["compare_rule"],
        "graphQuery": cfg["graph_query"].strip(),
    }

print(json.dumps(semantic_baseline, ensure_ascii=False, indent=2))

(WORKBENCH_DIR / "mission5_semantic_validation_pack.json").write_text(
    json.dumps(semantic_baseline, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("\nSaved:", WORKBENCH_DIR / "mission5_semantic_validation_pack.json")

print("\n[SEMANTIC_VALIDATION_SUBMISSION] 템플릿")
print("""[SEMANTIC_VALIDATION_SUBMISSION]
team=<팀명>
validatedAtKst=<YYYY-MM-DD HH:MM>
scenarioId=<A|B>
question=<질문>
path=<Entity->...->Entity>
baselineSqlRows=<행수>
graphQueryStatus=<코드 또는 N/A>
graphRows=<행수 또는 N/A>
comparison=<PASS|FAIL|N/A>
failReason=<사유 또는 ->
notes=<추가 메모>
[/SEMANTIC_VALIDATION_SUBMISSION]""")


In [ ]:
from datetime import datetime, timedelta, timezone

if "semantic_baseline" not in globals():
    raise RuntimeError("먼저 바로 위 semantic baseline 셀을 실행하세요.")

q1_meta = semantic_baseline.get("A")
if not q1_meta:
    raise RuntimeError("Q1 시나리오(A) baseline 정보가 없습니다.")

q1_graph_rows_proxy = q1_meta["baselineSqlRows"]
q1_fail_rows_proxy = max(int(round(q1_graph_rows_proxy * 0.33)), 1)

kst = timezone(timedelta(hours=9))
validated_at = datetime.now(kst).strftime("%Y-%m-%d %H:%M")

def make_submission_q1(team: str, graph_rows: int, comparison: str, fail_reason: str, notes: str) -> str:
    return f"""[SEMANTIC_VALIDATION_SUBMISSION]
team={team}
validatedAtKst={validated_at}
scenarioId=A
question=캠페인 유입 주문 중 결제 실패 주문은?
path=Campaign->CampaignAttribution->Order->Payment
baselineSqlRows={q1_meta['baselineSqlRows']}
graphQueryStatus=200
graphRows={graph_rows}
comparison={comparison}
failReason={fail_reason}
notes={notes}
[/SEMANTIC_VALIDATION_SUBMISSION]"""

q1_pass_log = make_submission_q1(
    team="Team Alpha",
    graph_rows=q1_graph_rows_proxy,
    comparison="PASS",
    fail_reason="-",
    notes="샘플 10건 키(campaign_id, order_id) 일치 확인",
)

q1_fail_log = make_submission_q1(
    team="Team Beta",
    graph_rows=q1_fail_rows_proxy,
    comparison="FAIL",
    fail_reason="관계 방향 오류(Order_has_Payment)",
    notes="관계 수정 후 refreshGraph 재실행 예정",
)

print("Q1 PASS 샘플 로그:\n")
print(q1_pass_log)
print("\nQ1 FAIL 샘플 로그:\n")
print(q1_fail_log)

q1_logs_payload = {
    "scenarioId": "A",
    "generatedAtKst": validated_at,
    "baseline": {
        "question": q1_meta["question"],
        "path": q1_meta["path"],
        "baselineSqlRows": q1_meta["baselineSqlRows"],
        "graphRowsPassProxy": q1_graph_rows_proxy,
        "graphRowsFailProxy": q1_fail_rows_proxy,
        "compareRule": q1_meta["compareRule"],
    },
    "logs": {
        "pass": q1_pass_log,
        "fail": q1_fail_log,
    },
}

json_out = WORKBENCH_DIR / "mission5_semantic_q1_sample_logs.json"
txt_out = WORKBENCH_DIR / "mission5_semantic_q1_sample_logs.txt"
json_out.write_text(json.dumps(q1_logs_payload, ensure_ascii=False, indent=2), encoding="utf-8")
txt_out.write_text(q1_pass_log + "\n\n" + q1_fail_log + "\n", encoding="utf-8")
print("\nSaved:", json_out)
print("Saved:", txt_out)


In [ ]:
from datetime import datetime, timedelta, timezone

if "semantic_baseline" not in globals():
    raise RuntimeError("먼저 바로 위 semantic baseline 셀을 실행하세요.")

q3_meta = semantic_baseline.get("B")
if not q3_meta:
    raise RuntimeError("Q3 시나리오(B) baseline 정보가 없습니다.")

q3_rowlevel_proxy_sql = """
    WITH customer_order_cnt AS (
        SELECT customer_id, COUNT(*) AS cnt
        FROM orders
        GROUP BY customer_id
    )
    SELECT p.promotion_type, o.order_id, o.customer_id,
           CASE WHEN coc.cnt >= 2 THEN 1 ELSE 0 END AS is_repeat
    FROM orders o
    JOIN order_promotions op ON o.order_id = op.order_id
    JOIN promotions p ON op.promotion_id = p.promotion_id
    JOIN customer_order_cnt coc ON o.customer_id = coc.customer_id;
"""

q3_graph_rows_proxy = len(run_query(q3_rowlevel_proxy_sql))
q3_fail_rows_proxy = max(int(round(q3_graph_rows_proxy * 0.86)), 1)

kst = timezone(timedelta(hours=9))
validated_at = datetime.now(kst).strftime("%Y-%m-%d %H:%M")

def make_submission(team: str, graph_rows: int, comparison: str, fail_reason: str, notes: str) -> str:
    return f"""[SEMANTIC_VALIDATION_SUBMISSION]
team={team}
validatedAtKst={validated_at}
scenarioId=B
question=프로모션 유형별 재구매율은?
path=Promotion->OrderPromotion->Order->Customer
baselineSqlRows={q3_meta['baselineSqlRows']}
graphQueryStatus=200
graphRows={graph_rows}
comparison={comparison}
failReason={fail_reason}
notes={notes}
[/SEMANTIC_VALIDATION_SUBMISSION]"""

q3_pass_log = make_submission(
    team="Team Gamma",
    graph_rows=q3_graph_rows_proxy,
    comparison="PASS",
    fail_reason="-",
    notes="promotion_type별 orders_cnt/repurchase_rate가 SQL 기준과 허용오차(±0.001) 내 일치",
)

q3_fail_log = make_submission(
    team="Team Delta",
    graph_rows=q3_fail_rows_proxy,
    comparison="FAIL",
    fail_reason="Order_receives_OrderPromotion 관계 누락",
    notes="누락 관계 보완 후 refreshGraph 재실행 예정",
)

print("Q3 PASS 샘플 로그:\n")
print(q3_pass_log)
print("\nQ3 FAIL 샘플 로그:\n")
print(q3_fail_log)

q3_logs_payload = {
    "scenarioId": "B",
    "generatedAtKst": validated_at,
    "baseline": {
        "question": q3_meta["question"],
        "path": q3_meta["path"],
        "baselineSqlRows": q3_meta["baselineSqlRows"],
        "graphRowsPassProxy": q3_graph_rows_proxy,
        "graphRowsFailProxy": q3_fail_rows_proxy,
        "compareRule": q3_meta["compareRule"],
    },
    "logs": {
        "pass": q3_pass_log,
        "fail": q3_fail_log,
    },
}

json_out = WORKBENCH_DIR / "mission5_semantic_q3_sample_logs.json"
txt_out = WORKBENCH_DIR / "mission5_semantic_q3_sample_logs.txt"
json_out.write_text(json.dumps(q3_logs_payload, ensure_ascii=False, indent=2), encoding="utf-8")
txt_out.write_text(q3_pass_log + "\n\n" + q3_fail_log + "\n", encoding="utf-8")
print("\nSaved:", json_out)
print("Saved:", txt_out)


In [ ]:
team = "Team-A"
workspace_id = "<WORKSPACE_ID>"
ontology_id = "<ONTOLOGY_ID>"

track2_handoff = f"""[TRACK2_WORKIQ_HANDOFF_PACKAGE]
team={team}
handoffAtKst=<YYYY-MM-DD HH:MM>
workspaceId={workspace_id}
ontologyId={ontology_id}
ontologyName=retail_track1_ontology_v1
entityCount={len(entities)}
relationshipCount={len(relationships_core)}
corePaths=Campaign->Order->Payment;Order->Shipment->Return;Promotion->Order->Margin
mappingHighlights=Order:orders.order_status;Payment:payments.payment_status;Shipment:shipments.shipment_status;Campaign:campaign_attribution.campaign_id;SupportTicket:support_tickets.ticket_reason
openIssues=GraphModel live query not run|environment dependency|use SQL baseline;Ontology API preview|contract may change|keep definition export;mapping coverage sample only|full review pending|complete mapping sheet
workiqKeys=캠페인명,SummerPush,VIPRetention,FlashWeek,BackToSchool;상품명,AeroPhone X,SmartWatch Pro,UltraBook 15,DailyTee Cotton;고객등급,Platinum
evidenceLinks=generated/workbench/mission5_mapping_review.json
[/TRACK2_WORKIQ_HANDOFF_PACKAGE]"""

handoff_path = WORKBENCH_DIR / "TRACK2_WORKIQ_HANDOFF_PACKAGE.txt"
handoff_path.write_text(track2_handoff, encoding="utf-8")
print(track2_handoff)
print("\nSaved:", handoff_path)


## 기술 연계 관점의 다음 단계

Track1 산출물은 아래 기술 계약으로 Track2 WorkIQ, Track3 WebIQ, Track4 FoundryIQ에 순서대로 전달됩니다.

1. Ontology 식별/모델 요약/핵심 경로를 인계해 WorkIQ 검색 키를 고정
2. 검증 결과를 품질 리스크로 전달해 Track2 점수화 기준에 반영
3. 논리 관계 경로로 Track3 WebIQ의 공개 확인 범위를 정하고, Track4 FoundryIQ 결합 질의의 정형 기준으로 사용

관련 문서:
- [WORKBOOK.md](./WORKBOOK.md)
- [PREREQUISITES.md](../track2/PREREQUISITES.md)
- [WORKBOOK.md](../track2/WORKBOOK.md)
- [PREREQUISITES.md](../track3/PREREQUISITES.md)
- [WORKBOOK.md](../track3/WORKBOOK.md)
- [PREREQUISITES.md](../track4/PREREQUISITES.md)
- [WORKBOOK.md](../track4/WORKBOOK.md)
